In [ ]:
import chromadb
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
import ollama
import re

## Create your chroma client & Select Model 
- we use persistence to store it in memory such that even when we restart the machine or device we still have the document in place

- **Context** is the text you paste into the prompt so the model can answer from your document.

In [ ]:
chroma_client = chromadb.PersistentClient(path="CHROMA")
MODEL = "llama3.2:1b"
CONTEXT  =""
QUESTION = ""
PERSONA = f""" 
Answer the question using only the context below.
If the context does not contain the answer, say "I don't know based on the document."

Context:
{CONTEXT}

Question: {QUESTION}"""


## Load your documents

In [ ]:
loader = PyPDFLoader("data/farming.pdf")
pages = loader.load()

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/SWDKMW+Helvetica', '/Encoding': IndirectObject(1422, 0, 1964071170080), '/FirstChar': 23, '/FontDescriptor': IndirectObject(1437, 0, 1964071170080), '/LastChar': 31, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [556, 556, 556, 556, 556, 556, 556, 556, 556]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/SWDKMW+Courier', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(1439, 0, 1964071170080), '/LastChar': 83, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [600, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 600, 0, 0, 600, 600, 600, 600, 600, 0, 600, 600, 600, 600, 0, 0, 0, 0, 0, 0, 0, 0, 600, 0, 0, 0, 0, 0, 0, 600, 0, 0, 0, 0, 600, 0, 0, 0, 0, 600]}, but is not installed. Consider installing fontTools i

# Cleaning text

In [ ]:
def cleanText(text):
    text = re.sub(r'-\n(\w)', r'\1', text)     # re-join words hyphenated across lines
    text = text.replace('•', '\n- ')           # bullets as their own lines
    text = re.sub(r'[ \t]+', ' ', text)        # collapse spaces/tabs only
    text = re.sub(r'\n{3,}', '\n\n', text)     # cap blank lines
    return text.strip()

In [ ]:
for page in pages: 
    page.page_content = cleanText(page.page_content) 

# Chunking

higher chunk size less granularity 
higher chunk overlap more repetition hence  try and reduce the chunk over lap size to get better context


In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 800, chunk_overlap= 100) # created an instance of the text splitter
chunks = splitter.split_documents(pages)

# Embeddings

In [ ]:
# functions & global variables 

def createEmbedding(text):
    response = ollama.embed(
    model='nomic-embed-text-v2-moe',
    input=text,
    )
    return response.embeddings[0]

In [ ]:
doc_embeddings = [] # this is our list of embeddings from out document

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding) 

In [ ]:
ids = [ f"farming:{i}" for i in range(len(chunks))]
documents = [chunk.page_content for chunk in chunks ]
metadatas = [{"source": chunk.metadata.get("source", " "),  }for chunk in chunks ]
embeddings = doc_embeddings


confirming the sizes of each element to assertain consistency

In [ ]:
print(len(ids), len(documents), len(metadatas), len(embeddings))  # all four equal
print(len(embeddings))                                         # single number, e.g. 768

9 9 9 9
9


In [ ]:
collection_research = chroma_client.get_or_create_collection(name="CHROMA")

In [ ]:
len(embeddings)

9

Find out the difference between *upsert* and *add* ad highlight the respective usecase

In [ ]:
collection_research.add(
    ids = ids, 
    documents = documents, 
    metadatas = metadatas, 
    embeddings =  embeddings 
)

In [ ]:
response = collection_research.query(
    query_texts=["benefits of crop rotation?"],
    n_results=2,
    query_embeddings = [createEmbedding("benefits of crop rotation?")],
    # include=["documents", "metadatas", "distances"],
)
 

In [ ]:
response

{'ids': [['10201979:0', 'farming:0']],
 'embeddings': None,
 'documents': [['Do you have a problem with: \n\n- Soil crusting \n\n- Cloddy soil \n\n- Water stress (too wet or too dry) for crops \n\n- Soil erosion \n\n- Soil compaction \n\n- Low yields \n \nA crop rotation can help to manage your soil and \nfertility, reduce erosion, improve your soil’s health, \nand increase nutrients available for crops. \nBenefits of Crop Rotations: \n \n\n- Improve crop yields \n\n- Improve the workability of the soil \n\n- Reduce soil crusting \n\n- Increase water available for plants \n\n- Reduce erosion and sedimentation \n\n- Recycle plant nutrients in the soil \n\n- Provide better distribution of labor during the \ncrop season by using different crops, planting \ndates, and harvest periods \n\n- Reduce fertilizer & insecticide inputs',
   'Do you have a problem with: \n\n- Soil crusting \n\n- Cloddy soil \n\n- Water stress (too wet or too dry) for crops \n\n- Soil erosion \n\n- Soil compaction \

In [ ]:
response["documents"]

[['Do you have a problem with: \n\n- Soil crusting \n\n- Cloddy soil \n\n- Water stress (too wet or too dry) for crops \n\n- Soil erosion \n\n- Soil compaction \n\n- Low yields \n \nA crop rotation can help to manage your soil and \nfertility, reduce erosion, improve your soil’s health, \nand increase nutrients available for crops. \nBenefits of Crop Rotations: \n \n\n- Improve crop yields \n\n- Improve the workability of the soil \n\n- Reduce soil crusting \n\n- Increase water available for plants \n\n- Reduce erosion and sedimentation \n\n- Recycle plant nutrients in the soil \n\n- Provide better distribution of labor during the \ncrop season by using different crops, planting \ndates, and harvest periods \n\n- Reduce fertilizer & insecticide inputs',
  'Do you have a problem with: \n\n- Soil crusting \n\n- Cloddy soil \n\n- Water stress (too wet or too dry) for crops \n\n- Soil erosion \n\n- Soil compaction \n\n- Low yields \n \nA crop rotation can help to manage your soil and \nfer

# Retreival 

Recall what we did on pure ollama chat? i.e ollama.chat(model.., messages..., options)? now basd on that we use and guide the model to use our vector DB as out source of truth

In [ ]:
#  query vector DB 
def retreiveFromVectorDB(query, k):
    response = collection_research.query(
    query_texts=[query],
    n_results=k,
    query_embeddings = [createEmbedding(query)],
    # include=["documents", "metadatas", "distances"],
    )
    return response["documents"][0]

def answer(question, k=4):
    chunks = retreiveFromVectorDB(question, k=k)
    CONTEXT = "\n\n---\n\n".join(chunks)  
    QUESTION = question
    prompt = f"""Answer the question using only the context below.
If the context does not contain the answer, say "I don't know based on the document."

Context:
{CONTEXT}

Question: {QUESTION}"""

    resp = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},   # keeps answers grounded and repeatable (lower chance of hallucination and creativity)
    )
    return resp.message.content


In [ ]:
print(answer("benefits of crop rotation?"))

Based on the context, the benefits of crop rotation are:

- Improve crop yields 
- Improve the workability of the soil 
- Reduce soil crusting 
- Increase water available for plants 
- Reduce erosion and sedimentation 
- Recycle plant nutrients in the soil 
- Provide better distribution of labor during the crop season 
- Reduce fertilizer & insecticide inputs


**Find out:** How do temperature and model choice affect a RAG system's answers?